# Sentinel-1 InSAR baseline stack

The *only* capability that justifies a separate `asf` backend (rather than reusing `earthlens.earthdata`): building an **InSAR baseline stack** from a reference granule, with perpendicular- and temporal-baseline windows enforced server-side. This walkthrough runs the search anonymously (no EDL credentials needed) and inspects the resulting stack.

Needs the `[asf]` extra (`pip install earthlens[asf]`).

## Pick a reference granule

An InSAR stack is built *around* a reference scene — every other product in the stack is a candidate secondary acquisition, characterised by its perpendicular and temporal baselines relative to the reference. The reference must be a stackable product (`sentinel-1-slc`, `sentinel-1-burst`, `alos-palsar-slc`, …).

We use a known-good Sentinel-1A SLC over the Iceland rift zone — Sentinel-1 ascends regularly there, so the stack is dense.

In [ ]:
from earthlens.earthlens import EarthLens

REFERENCE = 'S1A_IW_SLC__1SDV_20240601T072115_20240601T072143_054132_06960B_6FE8'

el = EarthLens(
    data_source='insar',                       # alias for 'asf'
    variables=['sentinel-1-slc'],
    reference=REFERENCE,                       # <-- switches to stack mode
    perpendicular_baseline=(-100.0, 100.0),    # metres, (min, max)
    temporal_baseline=(0, 60),                 # whole days, (min, max)
    start='2024-01-01',                        # advisory in stack mode
    end='2024-12-31',
    path='insar_stack',
)
el.datasource._mode

## Run the stack search

Stack mode runs anonymously — `_search()` calls `asf_search.granule_search([reference])` to fetch the reference scene, then `.stack(opts=ASFSearchOptions(minBaselinePerp, maxBaselinePerp, temporalBaselineDays))` to fetch the windowed acquisitions. The SDK enforces the windows server-side.

In [ ]:
products = el.datasource._search()
print(f'{len(products)} acquisition(s) in the baseline window')

## Inspect the baselines

Each stacked product carries `perpendicularBaseline` (metres) and `temporalBaseline` (days, signed relative to the reference) in its metadata. The reference itself is in the stack with baselines `(0, 0)`.

In [ ]:
print(f'{"scene":62s} {"perp (m)":>10s} {"temp (d)":>10s}')
for product in products:
    perp = product.metadata.get('perpendicularBaseline')
    temp = product.metadata.get('temporalBaseline')
    print(f'{product.id:62s} {perp!s:>10s} {temp!s:>10s}')

## Tighter windowing

If the unfiltered stack returns too many acquisitions, narrow the perpendicular- or temporal-baseline window. The SDK pushes both into the catalog query, so the latency is unchanged.

In [ ]:
tight = EarthLens(
    data_source='insar',
    variables=['sentinel-1-slc'],
    reference=REFERENCE,
    perpendicular_baseline=(-50.0, 50.0),  # tighter than before
    temporal_baseline=(0, 24),             # 24-day repeat cycle
    start='2024-01-01',
    end='2024-12-31',
    path='insar_stack_tight',
)
tight_products = tight.datasource._search()
print(f'tight stack: {len(tight_products)} acquisition(s)')

## Downstream processing

The backend retrieves SAR products; it does not form interferograms. Hand the downloaded stack to a dedicated InSAR tool:

- [HyP3](https://hyp3-docs.asf.alaska.edu/) — ASF's hosted on-demand InSAR / RTC service.
- [ISCE2 / ISCE3](https://github.com/isce-framework/isce2) — NASA / JPL's InSAR Scientific Computing Environment.
- [SNAP](https://step.esa.int/main/toolboxes/snap/) — ESA's Sentinel application platform.
- [MintPy](https://mintpy.readthedocs.io/) — time-series InSAR analysis.

To actually fetch the products, set EDL credentials and call `el.download()` (see [Authentication](../../reference/asf/authentication.md)). The download is idempotent — a re-run skips files already on disk.